# Controls 96^3: branch ablation, fusion ablation and P baseline

Ablation notebook for the 3-branch model at **96^3** (storage raw 200^3, views from raw 200^3 at 224).
Every variant is **retrained from scratch** on the same split/seed/protocol (no test-time masking) for a
fixed 20-epoch budget with early stopping disabled (`PATIENCE=EPOCHS+1`); the best checkpoint by validation
AUC is kept, then evaluated on the calibrated test report. Seed set: `(42, 43, 44)`; results are aggregated
as mean +/- std per spec.

| Code | Configuration | `use_3d` | `n_2d` | `view_indices` | `fusion` | `gate_fixed` |
|---|---|---|---|---|---|---|
| P | 3 branches + full CrossGate (main) | True | 2 | (0, 1) | crossgate | False |
| B1 | ResNeXt3D + MaxViT Slab MIP (drop Full AIP) | True | 1 | (0,) | crossgate | False |
| B2 | ResNeXt3D + MaxViT Full AIP (drop Slab MIP) | True | 1 | (1,) | crossgate | False |
| B3 | 2xMaxViT (Slab MIP + Full AIP) + Concat (2D-only) | False | 2 | (0, 1) | concat | - |
| C1 | 3 branches + Concat + classifier | True | 2 | (0, 1) | concat | - |
| C2 | 3 branches + cross-attention, gate fixed = 1 | True | 2 | (0, 1) | crossgate | True |
| B4 | Full 3-branch model on Bilateral storage, 3D input reduced to 96 | True | 2 | (0, 1) | crossgate | False |

Views: index 0 = `slab_mip`, index 1 = `aip_full`. B1/B2 keep the projection and CrossGate but the fusion
block only has one 2D token. C2 removes the learned gate parameter (attention + residual add unchanged).

B4 uses the Bilateral 200^3 export projected into the same 224 2D views, keeps the full CrossGate model, and
resizes only the 3D branch input to 96 on the fly; it is retrained from scratch with a pretrained 2D backbone
(no raw warm-start). It isolates downscaling the Bilateral-200 representation to the raw-96 model input.

`CTRL_SMOKE=1` runs tiny synthetic CPU data for all specs/one seed/one epoch. Real runs need a GPU plus
`WANDB_API_KEY` and `HF_TOKEN`. All artifacts go local + Drive. Smoke uses `SmokeModel`, not the real
architectures: it validates orchestration, not architecture performance. Run cells top to bottom.
The full study is 21 runs of exactly 20 epochs each; use measured per-spec epoch seconds to estimate runtime,
not an assumed A100 speed. X-AI and information analysis stay off as an explicit control-sweep budget exception.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SMOKE = os.environ.get('CTRL_SMOKE', '0') == '1'
if not SMOKE and 'google.colab' in sys.modules:
    repo = Path('/content/glaucoma-thesis')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'wandb', 'scipy', 'matplotlib', 'huggingface_hub', 'python-dotenv'], check=True)
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
sys.path.insert(0, str(Path.cwd()))
import ast, gc, json, math, random, tempfile, time
import numpy as np
import torch
import torch.nn.functional as F
import wandb
import matplotlib
matplotlib.use('Agg')
from scripts import controls_data as cd, controls_model as cm, controls_training as ct, final_model as fm, final_training as ft
ft.load_env_file()
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
if SMOKE:
    torch.set_num_threads(1)

## Configuration
`SPECS` is the only source of truth for the variants; model kwargs are recorded in the run config so resume
and comparisons are exact. `EPOCHS=20` for every spec with early stopping disabled (`PATIENCE=EPOCHS+1`) so
all variants run the full fixed budget; the best-by-val-AUC checkpoint is still kept.
`RUN_TARGET` blank selects all specs/seeds; set it to a tag (e.g. `P_s42`) for a single recovery run.
`AUTO_BATCH` probes exact divisors of effective batch 16 (4 in smoke), including 1, at the actual AMP/accumulation.
Budget defaults to 90% of detected GPU memory in decimal GB, not GiB. Existing identities always reuse saved batches.
Old FP16 runs require explicit `CTRL_AMP_DTYPE=float16` and their original group; AMP is never silently changed.
Per-spec microbatches can affect BatchNorm/stochastic paths: fixed effective batch is not bitwise equivalence.
Changing effective batch requires a new group AND an explicitly approved new protocol.
Use `CTRL_RESUME=1` for recovery; `CTRL_EXTEND_EPOCHS` defaults to 0 and requires explicit training authorization.

In [ ]:
# ===== RUN IDENTITY & RESUME (edit for a new experiment or explicit recovery) =====
DEFAULT_RUN_GROUP = 'crossgate_controls_96_7spec'
RUN_GROUP = os.environ.get('CTRL_RUN_GROUP', DEFAULT_RUN_GROUP)
RUN_TARGET = os.environ.get('CTRL_RUN_TARGET', '')
RESUME = os.environ.get('CTRL_RESUME', '0') == '1'
EXTEND_EPOCHS = int(os.environ.get('CTRL_EXTEND_EPOCHS', '0'))
# ===== STAGE SWITCHES (set only approved stages; budget exception) =====
RUN_XAI = False
RUN_INFO = False
# ===== DATA SOURCES (edit locations, not the raw-data protocol) =====
HF_DATA_REPO = os.environ.get('HF_DATA_REPO', 'tqhuyen/harvard-oct-glaucoma-200')
BILATERAL_REPO = os.environ.get('CTRL_BILATERAL_REPO', 'tqhuyen/harvard-oct-glaucoma-200-bilateral')
BILATERAL_REVISION = os.environ.get('CTRL_BILATERAL_REVISION', '47632c96b206707fd6423ee5b4da159069f63eaf')
BILATERAL_EXPORT_ROOT = Path(os.environ.get('CTRL_BILATERAL_EXPORT_ROOT', '/content/final_dn_export'))
# ===== FROZEN STUDY CONFIG (do not edit or retune) =====
CHECKPOINT_EVERY_STEPS = 10
SPECS = [
    {'code': 'P', 'label': '3 branches + CrossGate', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'crossgate', 'gate_fixed': False},
    {'code': 'B1', 'label': 'ResNeXt3D + slab_mip', 'use_3d': True, 'n_2d': 1, 'view_indices': (0,), 'fusion': 'crossgate', 'gate_fixed': False},
    {'code': 'B2', 'label': 'ResNeXt3D + aip_full', 'use_3d': True, 'n_2d': 1, 'view_indices': (1,), 'fusion': 'crossgate', 'gate_fixed': False},
    {'code': 'B3', 'label': '2D-only + concat', 'use_3d': False, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'concat', 'gate_fixed': False},
    {'code': 'C1', 'label': '3 branches + concat', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'concat', 'gate_fixed': False},
    {'code': 'C2', 'label': '3 branches + attention gate=1', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'crossgate', 'gate_fixed': True},
    {'code': 'B4', 'label': 'Bilateral 200 -> 96 full model', 'use_3d': True, 'n_2d': 2, 'view_indices': (0, 1), 'fusion': 'crossgate', 'gate_fixed': False, 'dataset': 'bilateral'},
]
SEEDS = [int(s) for s in os.environ.get('CTRL_SEEDS', '42' if SMOKE else '42,43,44').split(',')]
STORE_RES = 8 if SMOKE else 200
RES3D = 8 if SMOKE else 96
RES2D = 8 if SMOKE else 224
D_LATENT, ENC2D = 256, 'maxvit_tiny_rw_224'
ENC3D_FEATURES = (32, 64, 128, 192)
EPOCHS = int(os.environ.get('CTRL_EPOCHS', '1' if SMOKE else '20'))
LR, WD = 1e-4, 1e-4
PATIENCE = EPOCHS + 1
# ===== RUNTIME BATCH & AMP (edit hardware settings; keep effective batch fixed) =====
BS = int(os.environ.get('CTRL_BS', '2'))
GRAD_ACCUM = int(os.environ.get('CTRL_GRAD_ACCUM', '2' if SMOKE else '8'))
AUTO_BATCH = os.environ.get('CTRL_AUTO_BATCH', '0' if SMOKE else '1') == '1'
GPU_TOTAL_GB = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9 if DEVICE.type == 'cuda' else 0.0
TARGET_VRAM_GB = float(os.environ.get('CTRL_TARGET_VRAM_GB', str(0.9 * GPU_TOTAL_GB)))
EFFECTIVE_BATCH = int(os.environ.get('CTRL_EFFECTIVE_BATCH', '4' if SMOKE else '16'))
AMP_DTYPE = os.environ.get('CTRL_AMP_DTYPE', 'bfloat16' if DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported() else 'float16')
NUM_WORKERS = int(os.environ.get('CTRL_NUM_WORKERS', '0' if SMOKE or os.name == 'nt' else '4'))
# ===== DERIVED PATHS & DRIVE (do not edit; use location environment overrides) =====
SPLITS = ('Training', 'Validation', 'Test')
HF_DATA_PATTERNS = [f'{split}_{kind}.npy' for split in SPLITS for kind in ('volumes', 'labels')]
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix='ctrl_smoke_')) if SMOKE else None
DATA_ROOT = SMOKE_ROOT / 'data' if SMOKE else Path(os.environ.get('CTRL_DATA_ROOT', '/content/final_data'))
LOCAL_ROOT = SMOKE_ROOT / 'runs' if SMOKE else Path('outputs/crossgate_controls_96') / RUN_GROUP
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', '/content/drive'))
DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/MasterBKDN/Thesis'))
DRIVE_DIR = DRIVE_ROOT / 'crossgate_controls_96' / RUN_GROUP

### Configuration validation and storage setup
Headless real runs must mount Drive externally and set `DRIVE_MOUNT`/`DRIVE_ROOT`; a local folder is not a mount.

In [ ]:
def validate_controls_config():
    if not SMOKE and DEVICE.type != 'cuda':
        raise RuntimeError('Real controls training requires CUDA; use CTRL_SMOKE=1 for CPU tests')
    if AMP_DTYPE not in ('float16', 'bfloat16'):
        raise ValueError('CTRL_AMP_DTYPE must be float16 or bfloat16')
    if DEVICE.type == 'cuda' and AMP_DTYPE == 'bfloat16' and not torch.cuda.is_bf16_supported():
        raise ValueError('BF16 unsupported on this CUDA device; set CTRL_AMP_DTYPE=float16')
    if min(BS, GRAD_ACCUM, EPOCHS, CHECKPOINT_EVERY_STEPS) < 1 or NUM_WORKERS < 0:
        raise ValueError('Batch, accumulation, epochs/checkpoint steps must be positive; workers nonnegative')
    if os.name == 'nt' and NUM_WORKERS != 0:
        raise ValueError('Windows mmap requires CTRL_NUM_WORKERS=0')
    if EXTEND_EPOCHS < 0 or (EXTEND_EPOCHS and not RESUME):
        raise ValueError('CTRL_EXTEND_EPOCHS must be nonnegative and requires CTRL_RESUME=1')
    if EFFECTIVE_BATCH < 1:
        raise ValueError('Effective batch must be positive')
    if EFFECTIVE_BATCH != (4 if SMOKE else 16) and (SMOKE or RUN_GROUP == DEFAULT_RUN_GROUP):
        raise ValueError('New effective batch requires a new RUN_GROUP and approved new protocol')
    if BS > EFFECTIVE_BATCH or EFFECTIVE_BATCH % BS or (not AUTO_BATCH and BS * GRAD_ACCUM != EFFECTIVE_BATCH):
        raise ValueError('Batch must divide effective batch; manual BS * GRAD_ACCUM must equal target')
    if not math.isfinite(TARGET_VRAM_GB) or TARGET_VRAM_GB < 0 or (not SMOKE and TARGET_VRAM_GB <= 0):
        raise ValueError('VRAM budget must be finite and positive for CUDA')
    if not SMOKE and (SEEDS != [42, 43, 44] or EPOCHS != 20):
        raise ValueError('Frozen study requires seeds 42,43,44 and 20 epochs; use RUN_TARGET for recovery')
    if (LR, WD) != (1e-4, 1e-4) or PATIENCE != EPOCHS + 1:
        raise ValueError('Frozen study requires LR=1e-4, WD=1e-4, and PATIENCE=EPOCHS+1 (fixed 20-epoch budget)')
    if not SEEDS or len(set(SEEDS)) != len(SEEDS) or any(s < 0 or s >= 2**32 for s in SEEDS):
        raise ValueError('Seeds must be unique unsigned 32-bit integers')
    if not RUN_GROUP or Path(RUN_GROUP).name != RUN_GROUP:
        raise ValueError('RUN_GROUP must be a nonempty directory name')
validate_controls_config()
if DEVICE.type == 'cuda':
    TARGET_VRAM_GB = min(TARGET_VRAM_GB, GPU_TOTAL_GB * 0.9)
    print(f'[device] {torch.cuda.get_device_name(DEVICE)} | total={GPU_TOTAL_GB:.2f} GB (decimal) | budget={TARGET_VRAM_GB:.2f} GB | headroom={GPU_TOTAL_GB - TARGET_VRAM_GB:.2f} GB | AMP={AMP_DTYPE}')
for spec in SPECS:
    if spec['fusion'] not in ('crossgate', 'concat'):
        raise ValueError(f"Unknown fusion: {spec['fusion']}")
    if len(spec['view_indices']) != spec['n_2d']:
        raise ValueError(f"{spec['code']}: view_indices/n_2d mismatch")
    if spec['fusion'] == 'crossgate' and not spec['use_3d']:
        raise ValueError(f"{spec['code']}: CrossGate requires the 3D branch")
def resolve_hf_token():
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    return token

selected = [(spec, seed, f"{spec['code']}_s{seed}") for spec in SPECS for seed in SEEDS if not RUN_TARGET or RUN_TARGET == f"{spec['code']}_s{seed}"]
if not selected:
    raise ValueError('RUN_TARGET does not match SPECS/SEEDS')
BILATERAL_NEEDED = any(spec.get('dataset', 'raw') == 'bilateral' for spec, _, _ in selected)
if BILATERAL_NEEDED and not SMOKE and not resolve_hf_token():
    raise RuntimeError('Bilateral control requires HF_TOKEN in .env/environment or Colab Secrets')
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        if 'google.colab' not in sys.modules:
            raise RuntimeError('Headless: mount real Drive externally and set DRIVE_MOUNT/DRIVE_ROOT first')
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError('Drive must be a verified mounted filesystem, not a local directory')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = ft.Artifacts(LOCAL_ROOT, None if SMOKE else DRIVE_DIR, smoke=SMOKE)
DATA_STORAGE = ft.Artifacts(DATA_ROOT, None if SMOKE else DRIVE_DIR / 'data', smoke=SMOKE)
print('Device:', DEVICE, '| specs:', [spec['code'] for spec in SPECS], '| seeds:', SEEDS, '| runs:', len(selected))

## Data
D1 downloads only the declared splits from HF with `HF_TOKEN` (`allow_patterns`); D1b prepares the verified
Bilateral export only when B4 is selected; D2 verifies the raw (and Bilateral) storage; D3 builds/caches the
2D views + depth-axis once per split; D4 defines the dataset factory used by the training loop.

### D1 - Download (HF auth + allow_patterns)



In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in zip(SPLITS, (10, 6, 6)):
        np.save(DATA_ROOT / f'{split}_volumes.npy', rng.integers(0, 255, (n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8))
        np.save(DATA_ROOT / f'{split}_labels.npy', np.arange(n, dtype=np.int64) % 2)
    print('[data] smoke synthetic arrays ready at', DATA_ROOT)
else:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Authenticated HF download requires HF_TOKEN in .env/environment or Colab Secrets')
    print('[data] repo', HF_DATA_REPO, '| patterns', len(HF_DATA_PATTERNS))
    if not all((DATA_ROOT / name).is_file() for name in HF_DATA_PATTERNS):
        print('[data] downloading declared splits (authenticated)...')
        snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=token, allow_patterns=HF_DATA_PATTERNS)
        print('[data] download complete')
    else:
        print('[data] cache hit; no download needed')

### D1b - Bilateral export for B4 (cache-only, verified)
Runs only when a selected spec uses `dataset='bilateral'`. Authenticated selective download at the pinned
revision, verified against the export manifest, then imported without denoising or publication.



In [ ]:
BILATERAL_IDENTITY = None
if BILATERAL_NEEDED:
    if SMOKE:
        for split in SPLITS:
            raw = np.load(DATA_ROOT / f'{split}_volumes.npy')
            np.save(DATA_ROOT / f'{split}_volumes_dn.npy', raw.copy())
            ft.atomic_save({'source': ft.file_identity(DATA_ROOT / f'{split}_volumes.npy'), 'method': 'smoke', 'params': {}, 'implementation': 'smoke', 'shape': list(raw.shape), 'version': 2}, DATA_ROOT / f'{split}_volumes_dn.complete.pt')
        print('[data] smoke synthetic bilateral arrays ready')
    else:
        BILATERAL_IDENTITY = cd.prepare_bilateral(DATA_ROOT, BILATERAL_EXPORT_ROOT, token=resolve_hf_token(), repo=BILATERAL_REPO, revision=BILATERAL_REVISION, splits=SPLITS, res2d=RES2D, store_res=STORE_RES)
        print('[data] bilateral export', BILATERAL_IDENTITY['revision'], '| bilateral identity ready')

### D2 - Verify raw storage



In [ ]:
for split in SPLITS:
    volumes = np.load(DATA_ROOT / f'{split}_volumes.npy', mmap_mode='r')
    labels = np.load(DATA_ROOT / f'{split}_labels.npy')
    if volumes.ndim not in (4, 5) or (volumes.ndim == 5 and volumes.shape[1] != 1) or tuple(volumes.shape[-3:]) != (STORE_RES,) * 3:
        raise ValueError(f'Real training requires STORE_RES-cubed storage, got {volumes.shape[-3:]}')
    if volumes.dtype != np.uint8 or labels.ndim != 1 or not np.isin(labels, [0, 1]).all():
        raise ValueError(f'{split}: expected uint8 volumes and one-dimensional binary labels')
    if not len(volumes) or len(volumes) != len(labels):
        raise ValueError(f'{split}: volume/label count mismatch')
    print(f'[storage] {split}: {tuple(volumes.shape)} | labels {len(labels)} | pos {int(np.asarray(labels).sum())} | first-sample minmax={volumes[0].min() / 255:.4f}/{volumes[0].max() / 255:.4f}')
if BILATERAL_NEEDED:
    for split in SPLITS:
        dn = np.load(DATA_ROOT / f'{split}_volumes_dn.npy', mmap_mode='r')
        if dn.dtype != np.uint8 or tuple(dn.shape[-3:]) != (STORE_RES,) * 3 or len(dn) != len(np.load(DATA_ROOT / f'{split}_labels.npy')):
            raise ValueError(f'{split}: bilateral storage failed verification')
        print(f'[storage] {split} bilateral: {tuple(dn.shape)} | minmax={dn[0].min() / 255:.4f}/{dn[0].max() / 255:.4f}')

### D3 - View + depth-axis cache
Reuse source/projection-keyed caches locally and on Drive; no HF upload is authorized.



In [ ]:
for split in SPLITS:
    views_path, depth_path = ft.build_views(DATA_ROOT / f'{split}_volumes.npy', res2d=RES2D)
    print(f'[views] {split}: {views_path.name} + {depth_path.name}')
for split in SPLITS:
    for path in DATA_ROOT.glob(f'{split}_volumes_*{RES2D}*'):
        if '.partial.' not in path.name:
            DATA_STORAGE.sync(path)
print('[cache] view/depth caches synced')

### D4 - Dataset factory



In [ ]:
def make_datasets(ds, seed, spec):
    stem = 'volumes_dn' if ds == 'bilateral' else 'volumes'
    datasets = [ct.ControlsDataset(DATA_ROOT / f'{s}_{stem}.npy', DATA_ROOT / f'{s}_labels.npy', res3d=RES3D, res2d=RES2D, seed=seed, train=s == 'Training', use_3d=SMOKE or spec['use_3d']) for s in SPLITS]
    for split, dataset in zip(SPLITS, datasets):
        print(f'[dataset] {split}: n={len(dataset)} pos={int(np.asarray(dataset.labels).sum())} | res3d={RES3D} | views {RES2D}px | {dataset.source.name}')
    return datasets

## Train, evaluate, persist and explain
The training loop uses small factories: model init (per spec), batch/config resolution
(VRAM probe), val/test callbacks, and report/checkpoint/X-AI finalization. Metrics cadence: train per
optimizer step, val + test per epoch; calibrated `train/val/test` + CI + `report/split_table` at the end.


### T1 - Model (per spec, from scratch)



In [ ]:
def make_model(spec, tag_resume, artifacts):
    model_path = artifacts.local / 'best_weights.pt'
    if tag_resume or not model_path.is_file():
        model_path = None
    model = (ft.SmokeModel() if SMOKE else cm.ControlsModel(n_2d=spec['n_2d'], D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=not (tag_resume or model_path is not None), view_indices=spec['view_indices'], fusion=spec['fusion'], use_3d=spec['use_3d'], gate_fixed=spec['gate_fixed'])).to(DEVICE)
    if model_path is not None:
        ft.load_weights(model, model_path)
        print('[model] loaded weights from', model_path)
    else:
        print(f"[model] new model for {spec['code']} | fusion={spec['fusion']} views={spec['view_indices']} use_3d={spec['use_3d']} gate_fixed={spec['gate_fixed']}")
    return model

### T2 - Batch probe + config



In [ ]:
def resolve_batch(tag, tr, spec, seed, va, te):
    saved = ft.saved_config(*([LOCAL_ROOT / tag] if SMOKE else [LOCAL_ROOT / tag, DRIVE_DIR / tag]))
    if saved:
        if saved.get('amp_dtype', 'float16') != AMP_DTYPE:
            raise ValueError('Saved AMP differs: set CTRL_AMP_DTYPE to the original dtype for resume, or use a new CTRL_RUN_GROUP')
        bs, accum = int(saved['batch_size']), int(saved['grad_accum'])
        if bs < 1 or accum < 1 or bs * accum != EFFECTIVE_BATCH:
            raise ValueError('Saved effective batch differs from frozen protocol; use the original protocol/group')
        print('[batch] existing identity reuse bs/accum', bs, accum)
        return bs, accum, None
    if AUTO_BATCH and DEVICE.type == 'cuda':
        trials = []
        for candidate in reversed(range(1, EFFECTIVE_BATCH + 1)):
            if EFFECTIVE_BATCH % candidate:
                continue
            accum = EFFECTIVE_BATCH // candidate
            config = make_config(spec, seed, tr, va, te, candidate, accum)
            try:
                probe = ft.find_batch_size(lambda: ft.SmokeModel() if SMOKE else cm.ControlsModel(n_2d=spec['n_2d'], D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=False, view_indices=spec['view_indices'], fusion=spec['fusion'], use_3d=spec['use_3d'], gate_fixed=spec['gate_fixed']), tr, device=DEVICE, start=1, target_gb=TARGET_VRAM_GB, num_workers=NUM_WORKERS, config=config, candidates=[candidate], max_batch=EFFECTIVE_BATCH)
            except RuntimeError as exc:
                if not str(exc).startswith('No candidate batch fits the VRAM budget; trials='):
                    raise
                trials.extend(ast.literal_eval(str(exc).split('trials=', 1)[1]))
                continue
            probe['trials'] = trials + probe['trials']
            print('[batch] probe result (decimal GB):', probe)
            return candidate, accum, probe
        raise RuntimeError(f'No safe microbatch, including 1; trials={trials}')
    return BS, GRAD_ACCUM, None

def make_config(spec, seed, tr, va, te, bs, accum):
    ytr = tr.labels
    if set(np.unique(ytr)) != {0, 1}:
        raise ValueError('Training requires both binary classes')
    weights = [len(ytr) / (2 * int((ytr == c).sum())) for c in (0, 1)]
    return dict(dataset=spec.get('dataset', 'raw'), seed=seed, spec=spec['code'], fusion=spec['fusion'], view_indices=list(spec['view_indices']), use_3d=spec['use_3d'], gate_fixed=spec['gate_fixed'], epochs=EPOCHS, batch_size=bs, grad_accum=accum, lr=LR, weight_decay=WD, patience=PATIENCE, checkpoint_steps=CHECKPOINT_EVERY_STEPS, class_weights=weights, res3d=RES3D, res2d=RES2D, store_res=STORE_RES, n2d=spec['n_2d'], latent=D_LATENT, enc2d=ENC2D, enc3d_features=list(ENC3D_FEATURES), smoke=SMOKE, amp_dtype=AMP_DTYPE, torch_version=str(torch.__version__), device=str(DEVICE), data=[ft.data_identity(p) for d in (tr, va, te) for p in (d.source, d.label_path)])

def log_batch_probe(run, probe, bs, accum):
    run.summary.update({'batch/size': bs, 'batch/accum': accum, 'batch/target_gb': TARGET_VRAM_GB, 'device/name': torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'CPU smoke', 'device/total_decimal_gb': GPU_TOTAL_GB, 'device/headroom_decimal_gb': GPU_TOTAL_GB - TARGET_VRAM_GB, 'device/amp_dtype': AMP_DTYPE})
    if not probe:
        return
    table = wandb.Table(columns=['batch_size', 'peak_gb', 'ok'])
    for trial in probe['trials']:
        table.add_data(trial['batch_size'], trial['peak_gb'], trial['ok'])
    run.log({'report/batch_probe_table': table})
    run.summary.update({'batch/size': bs, 'batch/accum': accum, 'batch/peak_gb': probe['peak_gb'], 'batch/target_gb': TARGET_VRAM_GB})

### E1/E2 - Val/test callbacks (per-epoch metrics)



In [ ]:
METRIC_KEYS = ('loss', 'acc', 'balanced_acc', 'precision', 'recall', 'specificity', 'npv', 'f1', 'mcc', 'kappa', 'youden', 'auc_roc', 'auc_pr', 'ece', 'logloss', 'brier')

def metric_line(metrics):
    return ' '.join(f'{key}={metrics[key]:.4f}' for key in METRIC_KEYS if key in metrics)

def make_val_eval(va, bs, progress):
    def evaluate(model):
        p, y, logits = ft.predict(model, va, bs, num_workers=NUM_WORKERS, amp_dtype=AMP_DTYPE)
        metrics = {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        epoch = progress['trainer'].epoch + 1 if progress['trainer'] is not None else 0
        print(f'[val  ] epoch {epoch}: {metric_line(metrics)}', flush=True)
        return metrics
    return evaluate

def make_test_eval(te, bs, progress):
    def evaluate_test(model):
        p, y, logits = ft.predict(model, te, bs, num_workers=NUM_WORKERS, amp_dtype=AMP_DTYPE)
        metrics = {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        epoch = progress['trainer'].epoch if progress['trainer'] is not None else 0
        print(f'[test ] epoch {epoch}: {metric_line(metrics)}', flush=True)
        return metrics
    return evaluate_test


### E3/E4/E5 - Calibrated report, tables and X-AI scope
E3 fits calibration on validation only. E4 logs tables/scalars, never metric plots.
E5 is disabled under the explicit control-sweep budget exception; no extra analysis or training.



In [ ]:
def finish_tag(model, trainer, tr, va, te, artifacts, run, tag, seed, started, bs):
    res, probs, labels = ct.calibrated_report(model, va, te, bs, smoke=SMOKE, train=tr, num_workers=NUM_WORKERS, amp_dtype=AMP_DTYPE)
    res.update(tag=tag, seed=seed, hist=trainer.history, minutes=round((time.time() - started) / 60, 2))
    print(f'[eval] test AUC={res["test"]["auc_roc"]:.4f} F1={res["test"]["f1"]:.4f} | val AUC={res["val"]["auc_roc"]:.4f}')
    ft.log_report(run, res)
    params = sum(p.numel() for p in model.parameters())
    run.summary.update({'threshold': res['threshold'], 'temperature': res['temperature'], 'params': params, 'minutes': res['minutes']})
    weights_path = artifacts.save(ft.cpu_state(model), 'best_weights.pt')
    print('[checkpoint] saved', weights_path.name)
    ft.save_report(res, probs, labels, artifacts, run)
    if RUN_XAI:
        ft.save_xai(model, va, artifacts, run, smoke=SMOKE)
    return dict(res=res, weights_path=str(weights_path), params=params, test_probs=probs.tolist(), test_labels=labels.tolist())

### T3 - Training loop (per spec x seed)



In [ ]:
RESULTS = {}
ACTIVE_MODEL = ACTIVE_TRAINER = WANDB_RUN = None
LAST = {}
for spec, seed, tag in selected:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te = make_datasets(spec.get('dataset', 'raw'), seed, spec)
    bs, accum, batch_probe = resolve_batch(tag, tr, spec, seed, va, te)
    config = make_config(spec, seed, tr, va, te, bs, accum)
    artifacts = ft.Artifacts(LOCAL_ROOT / tag, None if SMOKE else DRIVE_DIR / tag, smoke=SMOKE)
    status = ft.run_status(artifacts, config, resume=RESUME, extend_epochs=EXTEND_EPOCHS)
    config = status['config']
    print(f'[run] {tag}: status={status["status"]} epochs={config["epochs"]}')
    LAST = {'tag': tag, 'artifacts': artifacts, 'config': config}
    if status['status'] == 'complete':
        RESULTS[tag] = {'res': status['result']}
        continue
    tag_resume = status['status'] == 'resume'
    WANDB_RUN = ft.init_wandb('ctrl_' + tag, config, artifacts, resume=status['status'] != 'new', smoke=SMOKE)
    stopped, started = False, time.time()
    exit_code = 1
    progress = {'trainer': None}
    try:
        log_batch_probe(WANDB_RUN, batch_probe, bs, accum)
        ACTIVE_MODEL = make_model(spec, tag_resume, artifacts)
        ACTIVE_TRAINER = ct.Trainer(ACTIVE_MODEL, tr, config, artifacts, WANDB_RUN, resume=tag_resume, num_workers=NUM_WORKERS)
        progress['trainer'] = ACTIVE_TRAINER
        print(f'[train] start {tag}: epochs={config["epochs"]} bs={bs} accum={accum} eff={bs * accum} workers={NUM_WORKERS} cuda={DEVICE.type == "cuda"}')
        timer = {'epoch': ACTIVE_TRAINER.epoch, 'started': time.monotonic(), 'offset': ACTIVE_TRAINER.cursor}
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize(DEVICE)
            torch.cuda.reset_peak_memory_stats(DEVICE)
        def epoch_timer(trainer):
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize(DEVICE)
            now = time.monotonic()
            seconds = max(now - timer['started'], 1e-9)
            partial = timer['offset'] > 0
            samples = len(tr) - timer['offset']
            peak = torch.cuda.max_memory_reserved(DEVICE) / 2**30 if DEVICE.type == 'cuda' else 0.0
            trainer.run.log({'progress/step': trainer.step, 'train/epoch_seconds': seconds, 'train/samples_per_second': samples / seconds, 'train/epoch_partial_resume': partial, 'train/peak_reserved_gib': peak})
            print(f'[train] epoch {trainer.epoch} done in {seconds:.1f}s | {samples / seconds:.2f} samples/s (includes eval/checkpoint) | peak {peak:.2f} GiB' + (' | partial resume' if partial else ''), flush=True)
            timer.update(epoch=trainer.epoch, started=time.monotonic(), offset=0)
            if DEVICE.type == 'cuda':
                torch.cuda.reset_peak_memory_stats(DEVICE)
        stopped = not ACTIVE_TRAINER.fit(make_val_eval(va, bs, progress), test_evaluate=make_test_eval(te, bs, progress), epoch_hook=epoch_timer)
        print('[train] fit finished | stopped =', stopped)
        if not stopped:
            ACTIVE_MODEL.load_state_dict(ACTIVE_TRAINER.best_state)
            RESULTS[tag] = finish_tag(ACTIVE_MODEL, ACTIVE_TRAINER, tr, va, te, artifacts, WANDB_RUN, tag, seed, started, bs)
            ft.complete_run(artifacts, config, WANDB_RUN.id)
        WANDB_RUN.summary['status'] = 'stopped' if stopped else 'success'
        WANDB_RUN.summary['stopped_safely'] = stopped
        exit_code = 0
    finally:
        try:
            if exit_code:
                WANDB_RUN.summary['status'] = 'failure'
            WANDB_RUN.finish(exit_code=exit_code)
        finally:
            progress['trainer'] = None
            ACTIVE_TRAINER = ACTIVE_MODEL = WANDB_RUN = None
            gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    if stopped:
        print('Stopped safely. Set RESUME=True and RUN_TARGET to', tag)
        break
print('Completed:', list(RESULTS))

## Aggregate results (mean +/- std across seeds)
Reads every finished run from local + Drive, builds the per-spec aggregate table (mean/std/min/max for all
val/test metrics), writes `controls_96_summary.csv` / `.json` / `.pt`, logs the table to W&B and syncs Drive.


In [ ]:
rows = ft.publish_summary(STORAGE)
summary = {}
if rows:
    for row in rows:
        code = row['tag'].split('_s')[0]
        summary.setdefault(code, []).append(row)
if summary:
    metric_keys = sorted({key for row in rows for key in row if key.startswith(('val_', 'test_')) and key not in ('val_n', 'test_n')})
    agg = {}
    for code, items in summary.items():
        agg[code] = {'n_seeds': len(items), 'seeds': sorted(int(item['seed']) for item in items)}
        for key in metric_keys:
            values = np.asarray([item[key] for item in items], dtype=float)
            agg[code][key] = {'mean': float(values.mean()), 'std': float(values.std(ddof=1)) if len(values) > 1 else 0.0, 'min': float(values.min()), 'max': float(values.max())}
    order = [spec['code'] for spec in SPECS if spec['code'] in agg]
    lines = ['spec,n_seeds,' + ','.join(f'{key}_mean,{key}_std' for key in metric_keys)]
    for code in order:
        item = agg[code]
        lines.append(code + ',' + str(item['n_seeds']) + ',' + ','.join(f"{item[key]['mean']:.4f},{item[key]['std']:.4f}" for key in metric_keys))
    csv_path = STORAGE.local / 'controls_96_summary.csv'
    csv_path.write_text('\n'.join(lines))
    STORAGE.sync(csv_path)
    STORAGE.save(agg, 'controls_96_summary.pt')
    json_path = STORAGE.local / 'controls_96_summary.json'
    json_path.write_text(json.dumps(agg, indent=2), encoding='utf-8')
    STORAGE.sync(json_path)
    print('[summary] specs:', order)
    for code in order:
        item = agg[code]
        print(f"[summary] {code}: n={item['n_seeds']} test_auc_roc={item['test_auc_roc']['mean']:.4f}+-{item['test_auc_roc']['std']:.4f} test_f1={item['test_f1']['mean']:.4f}+-{item['test_f1']['std']:.4f}")
    ft.load_env_file()
    summary_run = wandb.init(project='glaucoma-thesis', name='controls_96_summary_' + time.strftime('%Y%m%d_%H%M%S'), dir=str(STORAGE.local), config={'run_group': RUN_GROUP, 'specs': order, 'amp_dtype': AMP_DTYPE, 'effective_batch': EFFECTIVE_BATCH, 'epochs': EPOCHS, 'seeds': SEEDS}, reinit=True, mode='offline' if SMOKE else 'online')
    summary_exit_code = 1
    try:
        table = wandb.Table(columns=['spec', 'n_seeds', 'test_auc_roc_mean', 'test_auc_roc_std', 'test_f1_mean', 'test_f1_std'])
        for code in order:
            item = agg[code]
            table.add_data(code, item['n_seeds'], item['test_auc_roc']['mean'], item['test_auc_roc']['std'], item['test_f1']['mean'], item['test_f1']['std'])
        summary_run.log({'report/controls_summary': table})
        summary_run.summary.update({'spec_count': len(order), 'run_count': len(rows)})
        summary_exit_code = 0
    finally:
        summary_run.finish(exit_code=summary_exit_code)
else:
    print('No completed runs; no aggregate table yet.')
print('Smoke verified only local/offline behavior.' if SMOKE else 'Aggregates synced to the verified Drive mount.')

## Outputs and limitations
- Runs: `outputs/crossgate_controls_96/<RUN_GROUP>/<spec>_s<seed>/` with `metrics.json`, `test_predictions.pt`
  and `best_weights.pt`; W&B runs `ctrl_<spec>_s<seed>` in project `glaucoma-thesis`.
- Aggregate: `controls_96_summary.csv` / `.json` / `.pt` + `report/controls_summary` table in W&B.
- Every spec is trained from scratch on the same split/seed/protocol; best checkpoint by validation AUC,
  early stop with the same patience. Single-spec differences near noise should be read with the per-seed std
  and the calibrated CIs (not accuracy alone).
- B3 reads cached views without loading/resizing raw 3D samples during training/evaluation; D3 still builds shared views once.
- B4 uses the pinned Bilateral 200^3 export, projects its own 224 views, and resizes only the 3D branch to 96 on the fly; it does not reuse raw 2D views and is retrained from scratch.
- All specs run the full 20-epoch budget (`PATIENCE=21`); the reported test metrics come from the best validation-AUC checkpoint.
- The FP16 fallback uses the frozen probe and can fail on initial scaler overflow; A100 defaults to BF16.
- Bilateral fine-tune / raw-vs-bilateral comparisons beyond B4 belong to the separate final notebook under its approved protocol.